In [9]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path


def parse_multirow_header(raw_df, header_rows, data_start_row):
    levels = []
    for r in header_rows:
        row = list(raw_df.iloc[r])
        filled, last = [], None
        for v in row:
            if pd.notna(v) and str(v).strip():
                last = str(v).strip().replace('\n', ' ')
            filled.append(last)
        levels.append(filled)

    col_names = []
    for i in range(len(levels[0])):
        parts = []
        for lv in levels:
            val = lv[i] if i < len(lv) else None
            if val and (not parts or val != parts[-1]):
                parts.append(val)
        col_names.append(' | '.join(parts))

    data = raw_df.iloc[data_start_row:].copy()
    data.columns = col_names
    return data


def detect_config(raw_df):
    """Auto-detect if first row is a title (not a real header)."""
    r0_val = str(raw_df.iloc[0, 0]).strip()
    is_title_row = r0_val.lower().startswith('table') and raw_df.iloc[0, 1:].isna().all()

    if is_title_row:
        # Check if row 1 has 3 levels of headers (some cells in rows 2+3 are non-null)
        r2_non_null = raw_df.iloc[2].notna().sum()
        r3_non_null = raw_df.iloc[3].notna().sum() if len(raw_df) > 3 else 0
        if r2_non_null > 2 and r3_non_null > 2:
            return {'header_rows': [1, 2, 3], 'data_start': 4}
        else:
            return {'header_rows': [1, 2], 'data_start': 3}
    else:
        return {'header_rows': [0, 1], 'data_start': 2}


def clean_province(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    if not s or s.lower() in ('nan', 'province', 'total', 'kingdom of cambodia'):
        return None
    return s


def merge_excel_to_csv(input_path: str, output_path: str = None):
    input_path = Path(input_path)
    if output_path is None:
        output_path = input_path.with_name(input_path.stem + '_merged.csv')
    output_path = Path(output_path)

    xl = pd.ExcelFile(input_path)
    all_dfs = []

    for sheet_name in xl.sheet_names:
        raw = pd.read_excel(xl, sheet_name=sheet_name, header=None)
        cfg = detect_config(raw)

        df = parse_multirow_header(raw, cfg['header_rows'], cfg['data_start'])

        # Province is always the first column
        prov_col = df.columns[0]
        df = df.rename(columns={prov_col: 'Province'})
        df['Province'] = df['Province'].apply(clean_province)
        df = df[df['Province'].notna()].set_index('Province')
        df = df.dropna(axis=1, how='all')

        # Prefix every column with table label to ensure uniqueness
        label = sheet_name.replace(' ', '_')
        df.columns = [f"{label} | {c}" for c in df.columns]

        all_dfs.append(df)
        print(f"  {sheet_name}: {df.shape[1]} cols, {df.shape[0]} rows")
        print(df.columns)

    combined = pd.concat(all_dfs, axis=1)
    combined.to_csv(output_path)

    print(f"\nDone! {combined.shape[0]} provinces x {combined.shape[1]} columns")
    print(f"Saved to: {output_path}")
    return combined

In [10]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2015-2016.xlsx'
    out = '2015-2016.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of Schools', 'Table_1 | Disadv. Schools',
       'Table_1 | Number of Classes', 'Table_1 | Classes in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee',
       'Table_2 | 2 - Shift Schools | Lycee',
       'Table_2 | Floating Schools | Lycee',
       'Table_2 | Schools in Pagoda | Lycee',
       'Table_2 | Attached Pre-Sch. | Lycee',
       'Table_2 | School

In [11]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2016-2017.xlsx'
    out = '2016-2017.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of | Schools', 'Table_1 | Disadv. | Schools',
       'Table_1 | Number of | Classes', 'Table_1 | Classes | in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 29 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee', 'Table_2 | 2 - Shift | Schools',
       'Table_2 | Floating | Schools', 'Table_2 | Schools | in Pagoda',
       'Table_2 | Attached | Pre-Sch.',
       'Table_2 | Schools Without | Water Sup.',
     

In [12]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2017-2018.xlsx'
    out = '2017-2018.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of | Schools', 'Table_1 | Disadv. | Schools',
       'Table_1 | Number of | Classes', 'Table_1 | Classes | in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee', 'Table_2 | 2 - Shift | Schools',
       'Table_2 | Floating | Schools', 'Table_2 | Schools | in Pagoda',
       'Table_2 | Attached | Pre-Sch.',
       'Table_2 | Schools Without | Water Sup.',
     

In [13]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2018-2019.xlsx'
    out = '2018-2019.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of | Schools', 'Table_1 | Disadv. | Schools',
       'Table_1 | Number of | Classes', 'Table_1 | Classes | in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee', 'Table_2 | 2 - Shift | Schools',
       'Table_2 | Floating | Schools', 'Table_2 | Schools | in Pagoda',
       'Table_2 | Attached | Pre-Sch.',
       'Table_2 | Schools Without | Water Sup.',
     

In [14]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2019-2020.xlsx'
    out = '2019-2020.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of | Schools', 'Table_1 | Disadv. | Schools',
       'Table_1 | Number of | Classes', 'Table_1 | Classes | in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee', 'Table_2 | 2 - Shift | Schools',
       'Table_2 | Floating | Schools', 'Table_2 | Schools | in Pagoda',
       'Table_2 | Attached | Pre-Sch.',
       'Table_2 | Schools Without | Water Sup.',
     

In [15]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2020-2021.xlsx'
    out = '2020-2021.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of | Schools', 'Table_1 | Disadv. | Schools',
       'Table_1 | Number of | Classes', 'Table_1 | Classes | in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee', 'Table_2 | 2 - Shift | Schools',
       'Table_2 | Floating | Schools', 'Table_2 | Schools | in Pagoda',
       'Table_2 | Attached | Pre-Sch.',
       'Table_2 | Schools Without | Water Sup.',
     

In [17]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2020-2021.xlsx'
    out = '2021-2022.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of | Schools', 'Table_1 | Disadv. | Schools',
       'Table_1 | Number of | Classes', 'Table_1 | Classes | in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee', 'Table_2 | 2 - Shift | Schools',
       'Table_2 | Floating | Schools', 'Table_2 | Schools | in Pagoda',
       'Table_2 | Attached | Pre-Sch.',
       'Table_2 | Schools Without | Water Sup.',
     

In [18]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2022-2023.xlsx'
    out = '2022-2023.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of Schools', 'Table_1 | Disadv. Schools',
       'Table_1 | Number of Classes', 'Table_1 | Classes in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee',
       'Table_2 | 2 - Shift Schools | Lycee',
       'Table_2 | Floating Schools | Lycee',
       'Table_2 | Schools in Pagoda | Lycee',
       'Table_2 | Attached Pre-Sch. | Lycee',
       'Table_2 | School

In [ ]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2015-2016.xlsx'
    out = '2015-2016.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of Schools', 'Table_1 | Disadv. Schools',
       'Table_1 | Number of Classes', 'Table_1 | Classes in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee',
       'Table_2 | 2 - Shift Schools | Lycee',
       'Table_2 | Floating Schools | Lycee',
       'Table_2 | Schools in Pagoda | Lycee',
       'Table_2 | Attached Pre-Sch. | Lycee',
       'Table_2 | School

In [19]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2023-2024.xlsx'
    out = '2023-2024.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 15 cols, 28 rows
Index(['Table_1 | Number of Schools', 'Table_1 | Disadv. Schools',
       'Table_1 | Number of Classes', 'Table_1 | Number of Classroom',
       'Table_1 | Classes in Pagoda', 'Table_1 | Enrollment | Total',
       'Table_1 | Enrollment | Girl', 'Table_1 | Repeaters | Total',
       'Table_1 | Repeaters | Girl', 'Table_1 | Teaching Staff | Total',
       'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-Sch.',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | College',
       'Table_2 | Number of Schools | Lycee',
       'Table_2 | 2 - Shift Schools | Lycee',
       'Table_2 | Floating Schools | Lycee',
       'Table_2 | Schools in Pagoda | Lycee',
       'Table_2 | Attached Pre

In [20]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2024-2025.xlsx'
    out = '2024-2025.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 13 cols, 28 rows
Index(['Table_1 | Number of Schools', 'Table_1 | Number of Classes',
       'Table_1 | Classes in Pagoda', 'Table_1 | Enrollment | Total',
       'Table_1 | Enrollment | Girl', 'Table_1 | Repeaters | Total',
       'Table_1 | Repeaters | Girl', 'Table_1 | Teaching Staff | Total',
       'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-School',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | L.Sec.Sch.',
       'Table_2 | Number of Schools | U.Sec.Sch.',
       'Table_2 | 2-Shift Schools | U.Sec.Sch.',
       'Table_2 | Floating Schools | U.Sec.Sch.',
       'Table_2 | Schools in Pagoda | U.Sec.Sch.',
       'Table_2 | Attached Pre-School | U.Sec.Sch.',
       'Table_2 | Schoo

In [21]:
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python merge_excel_to_csv.py <input.xlsx> [output.csv]")
        sys.exit(1)

    inp = '2025-2026.xlsx'
    out = '2025-2026.csv'
    merge_excel_to_csv(inp, out)

  Table 1: 14 cols, 28 rows
Index(['Table_1 | Number of Schools', 'Table_1 | Number of Classes',
       'Table_1 | Number of Classrooms', 'Table_1 | Classes in Pagoda',
       'Table_1 | Enrollment | Total', 'Table_1 | Enrollment | Girl',
       'Table_1 | Repeaters | Total', 'Table_1 | Repeaters | Girl',
       'Table_1 | Teaching Staff | Total', 'Table_1 | Teaching Staff | Female',
       'Table_1 | Non-Teaching Staff | Total',
       'Table_1 | Non-Teaching Staff | Female',
       'Table_1 | Total Staff | Total', 'Table_1 | Total Staff | Female'],
      dtype='object')
  Table 2: 14 cols, 28 rows
Index(['Table_2 | Number of Schools | Pre-School',
       'Table_2 | Number of Schools | Primary',
       'Table_2 | Number of Schools | L.Sec.Sch.',
       'Table_2 | Number of Schools | U.Sec.Sch.',
       'Table_2 | 2-Shift Schools | U.Sec.Sch.',
       'Table_2 | Floating Schools | U.Sec.Sch.',
       'Table_2 | Schools in Pagoda | U.Sec.Sch.',
       'Table_2 | Attached Pre-Schools | U